# EcoData · S9 — Observabilidad con Grafana y costos
**Lanzeloth Vargas Marichi** · Proyecto Sello EcoData · Big Data, UPeU

La guía instrumenta un pipeline de streaming con métricas de **infraestructura** (Kafka → kafka-exporter → Prometheus → Grafana) y de **aplicación** (campos calculados en Spark). EcoData ya tiene Prometheus, Grafana, Pushgateway y 8 alertas en producción (S12); este cuaderno genera la evidencia que pide S9 sobre esa misma infraestructura.

| Guía S9 | EcoData |
|---|---|
| topic `atmos-eventos` | **`ecodata-s09-lecturas`** (lecturas reales de PM2.5/PM10 de Juliaca, Open-Meteo) |
| grupo `uso-atmos-group` | **`ec-alerta-ms-group`** y **`validador-ecodata-group`** (creados en S6 y S7) |
| `isValid`, `processedAt`, `latencyMs` | iguales, calculados en Spark; `latencyMs` = proceso − **llegada a Kafka** |
| alertas propuestas (Tabla 8) | las 8 reglas **ya implementadas** en `alerts.yml`, más umbrales nuevos calculados con datos medidos |
| costos (Tabla 9) | plantilla con tus valores medidos y el costo de `operacion.md` |

> **Antes de empezar:** Kafka, Prometheus (49090), Grafana (43000) y Pushgateway deben estar levantados; haz S6 antes (crea los grupos de consumo que se ven en el lag). Para que Spark no se quede sin memoria, **cierra otros cuadernos** de Spark que tengas abiertos (Kernel → Shut down).
>
> **Transparencia:** los eventos de este cuaderno son horas reales de la API publicadas **a ritmo controlado** (1 cada 2 s) para poder observar latencia y rendimiento; no son un sensor en vivo.

## 0. Preparación

In [1]:
import sys, json, time, threading, urllib.request, urllib.parse, os
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import aire_comun as aire
import openmeteo_a_kafka as pub_aire
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"

def crear_topic(nombre, particiones=3):
    adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
    try: adm.create_topics([NewTopic(nombre, num_partitions=particiones, replication_factor=1)]); print("topic creado:", nombre)
    except TopicAlreadyExistsError: print("el topic ya existía:", nombre)
    adm.close()

def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception:
            continue
    return None

def publicar_paced(topic, eventos, pausa_s=2.0):
    """Publica un evento cada pausa_s segundos en segundo plano (la celda no se bloquea)."""
    def run():
        p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                          value_serializer=lambda v: json.dumps(v).encode())
        for e in eventos:
            p.send(topic, key=e["sensorId"], value=e); p.flush(); time.sleep(pausa_s)
        p.close()
    th = threading.Thread(target=run, daemon=True); th.start(); return th

def desde_archivo(path="/tmp/s12/aq_juliaca_2023.json"):
    d = json.load(open(path)); h = d.get("hourly", d); out = []
    for t, a, b in zip(h["time"], h["pm2_5"], h["pm10"]):
        if a is None: continue
        ts = int(datetime.fromisoformat(t).replace(tzinfo=timezone.utc).timestamp() * 1000)
        out.append((ts, float(a), float(b) if b is not None else float(a)))
    return out

TOPIC = "ecodata-s09-lecturas"
crear_topic(TOPIC)

topic creado: ecodata-s09-lecturas


## 1. Prometheus: targets y consultas PromQL (3.1–3.2)
Verifica a mano en http://localhost:49090 → **Status → Targets** (`kafka-exporter`, `pushgateway` y `prometheus` en **UP**) y captura la pantalla. Luego corre las consultas desde aquí:

In [2]:
consultas = {
  "kafka_brokers": "kafka_brokers",
  "exporter arriba": 'up{job="kafka-exporter"}',
  "pushgateway arriba": 'up{job="pushgateway"}',
  "lag ec-alerta-ms-group": 'sum(kafka_consumergroup_lag{consumergroup="ec-alerta-ms-group"})',
  "lag validador-ecodata-group": 'sum(kafka_consumergroup_lag{consumergroup="validador-ecodata-group"})',
  "mensajes en atmos-aire": 'sum(kafka_topic_partition_current_offset{topic="atmos-aire"})',
}
for nombre, q in consultas.items():
    r = prom(q)
    print(f"{nombre:30s}", "SIN RESPUESTA de Prometheus" if r is None else (r[0]["value"][1] if r else "sin datos (¿el grupo ya existe? corre S6/S7)"))

kafka_brokers                  1
exporter arriba                1
pushgateway arriba             1
lag ec-alerta-ms-group         0
lag validador-ecodata-group    0
mensajes en atmos-aire         12983


**Tablero de Grafana (3.3):** importa `s09_dashboard_ecodata.json` (Dashboards → New → Import → Upload JSON file, datasource Prometheus). Trae los cuatro paneles de la guía (Kafka Brokers, Exporter Up, Broker Info, Consumer Lag) más lag por grupo y estado de los jobs de EcoData.

## 2. Spark: lectura streaming con campos de observabilidad (3.4–3.5)
`isValid` reutiliza el contrato mínimo; `processedAt` es el instante en que Spark procesa; `latencyMs = processedAt − kafkaTimestamp` mide cuánto tardó el evento desde que llegó al broker hasta Spark (el `timestamp` del evento es la **hora del dato** y puede ser de horas atrás, por eso no sirve para medir latencia del pipeline).

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType
esquema_evento = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
                  .add("pm25", DoubleType()).add("pm10", DoubleType())
                  .add("origen", StringType()).add("timestamp", LongType()))
spark = (SparkSession.builder.appName("ecodata-s09").master("local[2]")
         .config("spark.jars.packages", PKG)
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.port", "4050")          # distinto del 4040 que usan los jobs de S12
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7619816b-f24f-41d1-ad67-e8415a1f8edc;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in spark-list
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.

Spark 4.2.0


In [4]:
crudo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", "ecodata-s09-lecturas").option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos = (crudo.select("topic", "partition", "offset", F.col("timestamp").alias("kafkaTimestamp"),
                        F.from_json(F.col("value").cast("string"), esquema_evento).alias("e"))
           .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))
observables = (eventos
    .withColumn("isValid", F.col("tipoEvento").isNotNull() & F.col("sensorId").isNotNull() & F.col("pm25").isNotNull()
                           & F.col("pm10").isNotNull() & F.col("timestamp").isNotNull() & (F.col("pm25") >= 0))
    .withColumn("processedAt", F.unix_millis(F.current_timestamp()))
    .withColumn("latencyMs", F.col("processedAt") - F.unix_millis(F.col("kafkaTimestamp"))))
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## 3. Salida dual: Parquet + bitácora por micro-lote (3.6 y 3.8)
Cada micro-lote se guarda en Parquet y deja una línea en una bitácora con el formato de logs de la guía. (La consola de Spark no se ve en la celda de Jupyter, por eso la bitácora la reemplaza.)

In [5]:
RUTA = "./artifacts/s09_eventos_observabilidad"; CHK = "./artifacts/chk_s09"
bitacora = []
def por_microlote(df, bid):
    n = df.count()
    if n == 0:
        bitacora.append(dict(service="spark-streaming", component="observabilidad", batchId=bid, numInputRows=0, status="idle")); return
    df.write.mode("append").parquet(RUTA)
    r = df.agg(F.avg("latencyMs").alias("lat"), F.sum((~F.col("isValid")).cast("int")).alias("inval")).first()
    bitacora.append(dict(service="spark-streaming", component="observabilidad", topic="ecodata-s09-lecturas", batchId=bid,
                         numInputRows=n, avgLatencyMs=round(r["lat"], 1), invalidos=int(r["inval"]), status="processed"))

consulta = (observables.writeStream.foreachBatch(por_microlote)
            .option("checkpointLocation", CHK).trigger(processingTime="5 seconds").start())
print("consulta activa:", consulta.isActive)

consulta activa: True


Con la consulta activa, **publica el flujo** (45 lecturas reales, una cada 2 s ≈ 90 s). No detengas la consulta todavía.

In [7]:
serie = pub_aire.desde_api()                                   # [(ts_ms, pm25, pm10)]
paced = [aire.evento(t, a, b, "openmeteo-cams") for t, a, b in (serie * 3)[-45:]]
hilo = publicar_paced(TOPIC, paced, pausa_s=2.0)
print(len(paced), "eventos en camino hacia", TOPIC)

45 eventos en camino hacia ecodata-s09-lecturas


## 4. lastProgress: throughput y lag del lado de Spark (3.7)
Espera ~40 s desde la celda anterior y corre esta celda **dos o tres veces**.

In [8]:
p = consulta.lastProgress
if p is None:
    print("Todavía no hay micro-lote procesado. Espera unos segundos y repite.")
else:
    m = p["sources"][0].get("metrics", {})
    print(json.dumps({"service": "spark-streaming", "component": "observabilidad", "topic": "ecodata-s09-lecturas",
        "batchId": p["batchId"], "numInputRows": p["numInputRows"], "inputRowsPerSecond": p["inputRowsPerSecond"],
        "processedRowsPerSecond": p["processedRowsPerSecond"], "avgOffsetsBehindLatest": m.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": m.get("maxOffsetsBehindLatest"),
        "status": "idle" if p["numInputRows"] == 0 else "processed"}, indent=2))
print("\nÚltimas líneas de la bitácora:")
for b in bitacora[-5:]: print(json.dumps(b))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "ecodata-s09-lecturas",
  "batchId": 7,
  "numInputRows": 15,
  "inputRowsPerSecond": 3.0,
  "processedRowsPerSecond": 6.5,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}

Últimas líneas de la bitácora:
{"service": "spark-streaming", "component": "observabilidad", "topic": "ecodata-s09-lecturas", "batchId": 3, "numInputRows": 3, "avgLatencyMs": 2701.3, "invalidos": 0, "status": "processed"}
{"service": "spark-streaming", "component": "observabilidad", "topic": "ecodata-s09-lecturas", "batchId": 4, "numInputRows": 2, "avgLatencyMs": 2688.0, "invalidos": 0, "status": "processed"}
{"service": "spark-streaming", "component": "observabilidad", "topic": "ecodata-s09-lecturas", "batchId": 5, "numInputRows": 3, "avgLatencyMs": 2673.3, "invalidos": 0, "status": "processed"}
{"service": "spark-streaming", "component": "observabilidad", "topic": "ecodata-s09-lecturas", "bat

Evidencia alternativa: **Spark UI** en http://localhost:4050 → *Structured Streaming* (el puerto 4050 evita chocar con los jobs de S12, que usan 4040; si no abre desde tu navegador, usa el bloque de arriba).

In [9]:
hilo.join(timeout=120)          # espera a que termine la publicación
time.sleep(12)                  # y a que Spark procese el último micro-lote
consulta.stop()
guardado = spark.read.parquet(RUTA)
print("filas guardadas:", guardado.count())
guardado.select("partition", "offset", "sensorId", "pm25", "isValid", "latencyMs").orderBy(F.desc("offset")).show(10, truncate=False)

filas guardadas: 90
+---------+------+----------------------+----+-------+---------+
|partition|offset|sensorId              |pm25|isValid|latencyMs|
+---------+------+----------------------+----+-------+---------+
|1        |89    |openmeteo-cams-juliaca|2.1 |true   |4729     |
|1        |88    |openmeteo-cams-juliaca|2.0 |true   |1735     |
|1        |87    |openmeteo-cams-juliaca|1.7 |true   |3739     |
|1        |86    |openmeteo-cams-juliaca|1.4 |true   |744      |
|1        |85    |openmeteo-cams-juliaca|1.1 |true   |2750     |
|1        |84    |openmeteo-cams-juliaca|1.0 |true   |4755     |
|1        |83    |openmeteo-cams-juliaca|0.9 |true   |1761     |
|1        |82    |openmeteo-cams-juliaca|0.9 |true   |3765     |
|1        |81    |openmeteo-cams-juliaca|0.9 |true   |769      |
|1        |80    |openmeteo-cams-juliaca|0.8 |true   |2774     |
+---------+------+----------------------+----+-------+---------+
only showing top 10 rows


## 5. Latencia agregada por ventana de tiempo (3.9)

In [10]:
lat = (spark.read.parquet(RUTA).groupBy(F.window("kafkaTimestamp", "1 minute"))
       .agg(F.count("*").alias("numEventos"), F.avg("latencyMs").alias("avgLatencyMs"),
            F.min("latencyMs").alias("minLatencyMs"), F.max("latencyMs").alias("maxLatencyMs"))
       .select(F.col("window.start").alias("inicioVentana"), "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs")
       .orderBy("inicioVentana"))
lat.show(truncate=False)
EXPLICACION = ("El productor envía cada 2 s y Spark dispara cada 5 s: no son múltiplos (mcm = 10 s), así que la espera de "
               "cada evento hasta el siguiente disparo cicla entre ~0 y ~5 s. Esa es la variación de min/max: aritmética, no una falla.")
print(EXPLICACION)

+-------------------+----------+------------------+------------+------------+
|inicioVentana      |numEventos|avgLatencyMs      |minLatencyMs|maxLatencyMs|
+-------------------+----------+------------------+------------+------------+
|2026-11-06 02:50:00|20        |2533.55           |646         |4707        |
|2026-11-06 02:51:00|59        |2749.2542372881358|520         |4926        |
|2026-11-06 02:52:00|11        |2936.3636363636365|744         |4779        |
+-------------------+----------+------------------+------------+------------+

El productor envía cada 2 s y Spark dispara cada 5 s: no son múltiplos (mcm = 10 s), así que la espera de cada evento hasta el siguiente disparo cicla entre ~0 y ~5 s. Esa es la variación de min/max: aritmética, no una falla.


## 6. Alertas con umbrales justificados por datos medidos (3.10)

In [11]:
pl = spark.read.parquet(RUTA).select("latencyMs").toPandas()["latencyMs"]
p50, p95, mx = pl.quantile(.50), pl.quantile(.95), pl.max()
u_sens, u_crit = int(round(2 * p95, -2)), int(round(10 * p95, -2))
print(f"latencia medida: p50={p50:.0f} ms · p95={p95:.0f} ms · máx={mx:.0f} ms")
alertas = pd.DataFrame([
  ["Kafka no visible", "kafka_brokers < 1", "Prometheus", "ya implementada como PushgatewayCaido (caída del gateway); añadir esta para Kafka"],
  ["Exporter caído", 'up{job="kafka-exporter"} == 0', "Prometheus", "propuesta"],
  ["Lag alto sostenido", 'sum(kafka_consumergroup_lag{consumergroup="ec-alerta-ms-group"}) > 100 (for 2m)', "Prometheus", "propuesta; el lag normal medido en S6 vuelve a 0"],
  ["Latencia sensible", f"avgLatencyMs por minuto > {u_sens} ms (2×p95 medido)", "Spark / Parquet", "propuesta con tu p95"],
  ["Latencia crítica", f"avgLatencyMs por minuto > {u_crit} ms (10×p95 medido)", "Spark / Parquet", "propuesta con tu p95"],
  ["Sin eventos nuevos", "time() - push_time_seconds{job=lambda26_aire} > 600  or absent(...)", "Prometheus", "IMPLEMENTADA (alerts.yml): AireSinEventosNuevos / ClimaSinEventosNuevos"],
  ["Modelo no supera persistencia", "lambda26_aire_mejora_pct < 0 (10 m)", "Prometheus", "IMPLEMENTADA: AireModeloNoSuperaPersistencia"],
], columns=["Situación", "Regla", "Dónde se evalúa", "Estado"])
alertas

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


latencia medida: p50=2726 ms · p95=4820 ms · máx=4926 ms


,Situación,Regla,Dónde se evalúa,Estado
0,Kafka no visible,kafka_brokers < 1,Prometheus,ya implementada como PushgatewayCaido (caída d...
1,Exporter caído,"up{job=""kafka-exporter""} == 0",Prometheus,propuesta
2,Lag alto sostenido,"sum(kafka_consumergroup_lag{consumergroup=""ec-...",Prometheus,propuesta; el lag normal medido en S6 vuelve a 0
3,Latencia sensible,avgLatencyMs por minuto > 9600 ms (2×p95 medido),Spark / Parquet,propuesta con tu p95
4,Latencia crítica,avgLatencyMs por minuto > 48200 ms (10×p95 med...,Spark / Parquet,propuesta con tu p95
5,Sin eventos nuevos,time() - push_time_seconds{job=lambda26_aire} ...,Prometheus,IMPLEMENTADA (alerts.yml): AireSinEventosNuevo...
6,Modelo no supera persistencia,lambda26_aire_mejora_pct < 0 (10 m),Prometheus,IMPLEMENTADA: AireModeloNoSuperaPersistencia


## 7. Costos y plan de escalado (3.11)

In [12]:
import glob
bytes_parquet = sum(os.path.getsize(f) for f in glob.glob(RUTA + "/*.parquet"))
filas_parquet = spark.read.parquet(RUTA).count()
b_por_fila = bytes_parquet / max(filas_parquet, 1)
ritmo_lab = 0.5                               # eventos/s del laboratorio (1 cada 2 s)
ritmo_real = 2 / 3600                         # EcoData real: aire + clima, 1 evento/hora cada uno
def gb_mes(ev_s): return ev_s * b_por_fila * 3600 * 24 * 30 / 1e9
EC2_HORA, S3_GB_MES, INSTANCIAS = 0.0416, 0.023, 3     # t3.medium bajo demanda y S3 Standard (referencia de la guía; verifica el precio vigente)
computo = INSTANCIAS * EC2_HORA * 730
tabla = pd.DataFrame([
  ["Eventos por segundo (laboratorio, medido)", f"{ritmo_lab:.2f}", "inputRowsPerSecond de lastProgress (celda 4)"],
  ["Eventos por segundo (EcoData real)", f"{ritmo_real:.5f}", "1 evento/hora por variable"],
  ["Latencia promedio por minuto (medida)", f"{pl.mean():.0f} ms", "celda 5"],
  ["Bytes por fila en Parquet", f"{b_por_fila:.0f} B", f"{bytes_parquet:,} B / {filas_parquet} filas"],
  ["Crecimiento Parquet, laboratorio", f"{gb_mes(ritmo_lab):.2f} GB/mes", "si corriera 24/7 a este ritmo"],
  ["Crecimiento Parquet, EcoData real", f"{gb_mes(ritmo_real)*1000:.2f} MB/mes", "ritmo real"],
  ["Costo cómputo (3 instancias t3.medium)", f"US$ {computo:.2f}/mes", "3 × 0,0416 × 730 h (Kafka; Spark; Prometheus+Grafana)"],
  ["Costo almacenamiento (laboratorio)", f"US$ {gb_mes(ritmo_lab)*S3_GB_MES:.2f}/mes", "S3 Standard 0,023 US$/GB-mes"],
], columns=["Variable", "Valor medido / calculado", "Origen"])
tabla

,Variable,Valor medido / calculado,Origen
0,"Eventos por segundo (laboratorio, medido)",0.50,inputRowsPerSecond de lastProgress (celda 4)
1,Eventos por segundo (EcoData real),0.00056,1 evento/hora por variable
2,Latencia promedio por minuto (medida),2724 ms,celda 5
3,Bytes por fila en Parquet,1115 B,"100,394 B / 90 filas"
4,"Crecimiento Parquet, laboratorio",1.45 GB/mes,si corriera 24/7 a este ritmo
5,"Crecimiento Parquet, EcoData real",1.61 MB/mes,ritmo real
6,Costo cómputo (3 instancias t3.medium),US$ 91.10/mes,"3 × 0,0416 × 730 h (Kafka; Spark; Prometheus+G..."
7,Costo almacenamiento (laboratorio),US$ 0.03/mes,"S3 Standard 0,023 US$/GB-mes"


**Plan de escalado — escribe tu decisión con estos números** (ejemplo de estructura):
1. *Umbral 1:* si `kafka_consumergroup_lag` > 100 de forma sostenida (> 2 min) → subir particiones y consumidores.
2. *Umbral 2:* si la latencia promedio por minuto supera el valor crítico de la celda 6 → más paralelismo en Spark.
3. *Si el volumen se duplicara:* con el ritmo real de EcoData (2 eventos/hora) el cuello de botella no es el broker sino el modelo/Spark; se escalaría primero Spark. Justifícalo con tus medidas.

### Qué capturar (con **reloj del sistema y tu perfil visibles, sin recortar**)
1. Prometheus → Targets (UP) y las consultas de la sección 1.
2. Grafana: el tablero de S9 con los cuatro paneles.
3. Sección 2 (esquema con `isValid/processedAt/latencyMs`) y sección 4 (`lastProgress` + bitácora).
4. Sección 5 (latencia por minuto) y sección 6 (tabla de alertas).
5. Sección 7 (plantilla de costos) y tu texto del plan de escalado.

In [13]:
spark.stop()